In deze notebook wordt de feature uitgewerkt om de content in huisstijl te uploaden (DFIT-1261).
Zoals in ieder notebook beginnen we met de authenticatie om verbinding met de AskDelphi api te kunnen maken.

In [2]:
from ask_delphi_api.authentication import AskDelphiClient
client = AskDelphiClient()
client.authenticate()

Parsed tenant/project/acl from CMS URL
Loaded cached tokens.
AUTHENTICATION STARTED
Trying cached tokens...
Editing API token status: 401
Cached tokens failed: Failed to fetch editing API token:

Exchanging portal code...
Status: 200
Access token received.
Publication URL: https://digitalecoach.askdelphi.com
Tokens saved to cache.
Getting editing API token...
Editing API token status: 200
Editing API token acquired.
AUTHENTICATION SUCCESSFUL


True

Om de content opmaak aan te passen maken we eerst een nieuw topic aan, in dit geval een stap. Na het runnen van onderstaande code kun je in cms zien dat content inderdaad in huisstijl is opgemaakt.

In [ ]:
from ask_delphi_api.project import Project
from ask_delphi_api.topictools import TopicTools
from ask_delphi_api.workflow import Workflow

project = Project(client)
topics = TopicTools(client, project)
workflow = Workflow(client)

# voor een stap, vul je "Action" als tweede variabel in
topicId = topics.topic_upload("Nieuwe stap", "Action")

# Haal topic-parts op.
content = topics.get_topic_parts(topicId=topicId)

# Selecteer part uit topic met daarin de content.
body_part = None
groups = content['topicEditorData']['groups']
for group in groups:
    for part in group['parts']:
        if part["partId"] == "body":
            body_part = part

# Checkout topic.
result = topics.checkout(topicId)

# Selecteer huidig topicVersionId.
topicVersionId = result['topicVersionId']

text = f'<h1>Titel</h1><p>Inleiding met <strong>vet</strong> tekst.</p><ul><li>Eerste punt</li><li>Tweede punt</li></ul>'

# Pas content topic aan.
topics.topic_add_content(topicVersionId=topicVersionId, topicId=topicId, partId="body", part=body_part, new_text=text)

# Checkin topic.
topics.checkin(topicId)

# Topic publiceren
request_id = workflow.create_workflow_transition_request(topicId)
transitions_model = workflow.get_workflow_transition_request_transitions_model(request_id)
workflow.update_workflow_transition_request(request_id, transitions_model)
workflow.approve_workflow_transition_request(request_id)


2026-03-04T10:21:10Z


{}

Hieronder gaan we onderzoeken of het ook mogelijk is om een taaksjabloon uit een word document te kunnen uploaden waarbij de opmaak van de content wordt meegenomen.